# Bayesian Network Weather Forecasting
Uses the cleaned GSOD 2002 data to build and train a Bayesian Network.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/gsod_2002.csv', low_memory=False)
print(f'Loaded {len(df):,} records with {df.shape[1]} columns')
df.head()

## Discretize Continuous Variables
Bayesian Networks need discrete states. This is going to bin each continuous variable into weather categories.

In [ ]:
disc = pd.DataFrame()

# TEMP (°F): cold < 32, warm > 75, else avg
disc['TEMP_D'] = pd.cut(
    df['TEMP'],
    bins=[-np.inf, 32, 75, np.inf],
    labels=['cold', 'avg', 'warm']
)

# SLP: sea-level pressure (mbar): low < 1005, high >= 1005
disc['SLP_D'] = pd.cut(
    df['SLP'],
    bins=[-np.inf, 1005, np.inf],
    labels=['low', 'high']
)

# WDSP: mean wind speed (knots): calm < 5, moderate < 15, strong >= 15
disc['WIND_D'] = pd.cut(
    df['WDSP'],
    bins=[-np.inf, 5, 15, np.inf],
    labels=['calm', 'moderate', 'strong']
)

# DEWP: dew point (°F): low < 32, moderate < 55, high >= 55
disc['DEWP_D'] = pd.cut(
    df['DEWP'],
    bins=[-np.inf, 32, 55, np.inf],
    labels=['low', 'moderate', 'high']
)

# PRCP: precipitation (inches): none = 0, light < 0.5, heavy >= 0.5
disc['PRCP_D'] = pd.cut(
    df['PRCP'].fillna(0),
    bins=[-np.inf, 0.0, 0.5, np.inf],
    labels=['none', 'light', 'heavy']
)

# Binary event flags (already 0/1)
disc['FOG']     = df['FOG'].astype(int)
disc['RAIN']    = df['RAIN'].astype(int)
disc['SNOW']    = df['SNOW'].astype(int)
disc['THUNDER'] = df['THUNDER'].astype(int)

# Drop rows with any remaining NaN (this is from continuous sentinels)
disc.dropna(inplace=True)

# Convert categoric stuff to string so pgmpy can consume them cleanly
for col in ['TEMP_D', 'SLP_D', 'WIND_D', 'DEWP_D', 'PRCP_D']:
    disc[col] = disc[col].astype(str)

print(f'Discretized dataset: {len(disc):,} rows')
print(disc.dtypes)
disc.head()

In [ ]:
# distribution check of discretized variables
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, disc.columns):
    disc[col].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

for ax in axes[len(disc.columns):]:
    ax.set_visible(False)

plt.suptitle('Discretized Variable Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## Network Structure (the DAG)

Edge reasoning:
- **SLP -> TEMP**: pressure systems effect temperature patterns
- **SLP -> WIND**: pressure gradient creates wind
- **SLP -> PRCP**: low pressure brings precipitation
- **TEMP -> DEWP**: temperature controls dewpoint proximity
- **TEMP -> PRCP**: temperature determines rain vs snow
- **TEMP -> SNOW**: cold required for snow
- **DEWP -> FOG**: high dew point near temp -> fog
- **DEWP -> RAIN**: moisture content makes rain
- **PRCP -> RAIN**: precipitation type
- **PRCP -> SNOW**: precipitation type
- **WIND -> FOG**: wind disperses fog
- **TEMP -> THUNDER**: instability of warmth + moisture = thunder
- **DEWP -> THUNDER**: humidity helps thunderstorm development

In [ ]:
edges = [
    ('SLP_D',  'TEMP_D'),
    ('SLP_D',  'WIND_D'),
    ('SLP_D',  'PRCP_D'),
    ('TEMP_D', 'DEWP_D'),
    ('TEMP_D', 'PRCP_D'),
    ('TEMP_D', 'SNOW'),
    ('TEMP_D', 'THUNDER'),
    ('DEWP_D', 'FOG'),
    ('DEWP_D', 'RAIN'),
    ('DEWP_D', 'THUNDER'),
    ('PRCP_D', 'RAIN'),
    ('PRCP_D', 'SNOW'),
    ('WIND_D', 'FOG'),
]

model = DiscreteBayesianNetwork(edges)
print('Nodes:', model.nodes())
print('Edges:', model.edges())

## Train / Test Split and Parameter Learning here

In [ ]:
train_df, test_df = train_test_split(disc, test_size=0.2, random_state=42)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# Bayesian estimator with K2 (Dirichlet) prior to handle zero count cells
model.fit(
    train_df,
    estimator=BayesianEstimator,
    prior_type='K2'
)

print('\nModel trained successfully.')
print('CPD nodes:', [cpd.variable for cpd in model.cpds])

In [ ]:
# Inspecting rain, snow ,fog
for var in ['RAIN', 'SNOW', 'FOG']:
    cpd = model.get_cpds(var)
    print(cpd)
    print()

## Checking model here
Using variable elimination here

In [ ]:
infer = VariableElimination(model)

scenarios = [
    {'label': 'Cold + Low Pressure',    'evidence': {'TEMP_D': 'cold', 'SLP_D': 'low'}},
    {'label': 'Warm + High Pressure',   'evidence': {'TEMP_D': 'warm', 'SLP_D': 'high'}},
    {'label': 'Avg Temp + Strong Wind', 'evidence': {'TEMP_D': 'avg',  'WIND_D': 'strong'}},
    {'label': 'High Dew + Low Pressure','evidence': {'DEWP_D': 'high', 'SLP_D': 'low'}},
]

query_vars = ['RAIN', 'SNOW', 'FOG', 'THUNDER']

for scenario in scenarios:
    print(f"=== {scenario['label']} ===")
    for var in query_vars:
        result = infer.query(variables=[var], evidence=scenario['evidence'], show_progress=False)
        p = result.values
        labels = result.state_names[var]
        probs = {str(l): round(float(v), 4) for l, v in zip(labels, p)}
        print(f'  {var}: {probs}')
    print()

## Evaluation
Predicting and then comparing against actual

In [ ]:
# Evidence variables, everything except what we are predicting
EVIDENCE_COLS = ['SLP_D', 'TEMP_D', 'WIND_D', 'DEWP_D', 'PRCP_D']
TARGET_COLS   = ['RAIN', 'SNOW', 'FOG']

# Use a sample for speed if the test set is large
eval_df = test_df.sample(min(2000, len(test_df)), random_state=0).reset_index(drop=True)

predictions = {t: [] for t in TARGET_COLS}

for _, row in eval_df.iterrows():
    evidence = {col: row[col] for col in EVIDENCE_COLS if pd.notna(row[col])}
    for target in TARGET_COLS:
        try:
            q = infer.map_query(variables=[target], evidence=evidence, show_progress=False)
            predictions[target].append(q[target])
        except Exception:
            predictions[target].append(np.nan)

print('Evaluation complete.')

In [ ]:
for target in TARGET_COLS:
    pred = pd.array(predictions[target])
    actual = eval_df[target].values

    # Drop rows where prediction failed
    mask = pd.notna(pred)
    pred   = pred[mask].astype(int)
    actual = actual[mask].astype(int)

    print(f'=== {target} ===')
    print(classification_report(actual, pred, zero_division=0))

    cm = confusion_matrix(actual, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'pred_{i}' for i in range(cm.shape[1])],
                yticklabels=[f'true_{i}' for i in range(cm.shape[0])])
    plt.title(f'Confusion Matrix – {target}')
    plt.tight_layout()
    plt.show()
    print()

needs to be expanded a lot still but its a start on 2002 data.